# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FALAKNAZMALICK/MY-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

#**ML Task Type: Binary Classification (with secondary Scoring / Ranking)**

**Explanation**: We are training a classification model to output a probability score (between $0.0$ and $1.0$) indicating how likely a page is to be in a state of traffic decay. We can then rank all our pages from highest decay probability to lowest, allowing content editors to focus on the worst-decaying pages first.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target Variable ($y$)**: is_declining (Binary: 1 if the page's search trend is actively heading downward, 0 otherwise).

**Is it a proxy?** Yes, the label trend_direction == 'down' serves as our proxy. Because "decay" is a gradual and subjective business concept, we use a calculated statistical trend over the last 90 days as a concrete proxy to represent a decaying page.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary ML Metric: Precision at K (e.g., Precision@20) and Area Under the Precision-Recall Curve (PR-AUC).**

**Why this metric?** Since our dataset is likely imbalanced and content editors have limited hours, we care most about the top of our recommendation list. If we recommend 20 pages for a rewrite, we want as many of those 20 as possible to actually be decaying (high Precision). High Recall is a secondary worry—we don't need to catch every single decaying page instantly, but we absolutely must not waste editors' time with false alarms.

**Business Success Metric:** The reduction in traffic loss on updated pages over a 60-day post-update window compared to a control group of non-updated decaying pages.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [6]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
    REPO_DIR = "flyrank-ml-internship-starter"

    if not os.path.isdir(REPO_DIR):
        print("Cloning repository...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

    print("Setting working directory...")
    os.chdir(REPO_DIR)

    print("Installing requirements...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Current Working Directory:", os.getcwd())

Cloning repository...
Setting working directory...
Installing requirements...
Current Working Directory: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter


In [7]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'target_is_declining']


In [8]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["target_is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

display_cols = [
    "content_id",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "target_is_declining"
]

print("--- DataFrame Schema Snippet ---")
display(df[display_cols].head(3))

--- DataFrame Schema Snippet ---


,content_id,days_since_last_update,impressions_90d,clicks_90d,target_is_declining
0,content_304f48230142,20,3803,29,1
1,content_a1fb4e703a9e,25,15320,7,1
2,content_9aa793d4d895,20,12581,11,1


**The Unit of Analysis:** A single webpage (represented by page_id).

**What one row represents:** Each row in this dataframe represents a unique, tracked URL at a specific snapshot in time, along with its historical engagement features (clicks, impressions, age since last updated) and its corresponding binary decay target (target_is_declining)

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**The Flaw of Fixed Rules**: A simple heuristic like "Flag any page where days_since_last_update > 180" is highly inefficient. Some evergreen pages do not need updates for years and continue to gain traffic, while other highly seasonal pages might decay in just 30 days.

**How ML Wins**: An ML model can look at non-linear interactions across multiple features simultaneously—such as the interaction between age, recent impression drops, and click-through-rate (CTR) stability. This allows us to predict decay based on behavior rather than a naive timeline, preventing wasted editing hours.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.